# TickDB ETL Batch — HDFS + Múltiples Formatos

Lee datos desde Parquet (output del streaming), los transforma y los escribe en HDFS en **Parquet, Avro, ORC y Delta**.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
import os

spark = SparkSession.builder \
    .appName("TickDB-ETL-Batch") \
    .config("spark.jars.packages",
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,"
            "io.delta:delta-spark_2.12:3.0.0,"
            "org.apache.spark:spark-avro_2.12:3.5.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark: {spark.version}")

---
## 1. Leer desde Parquet (output del stream)

In [ ]:
INPUT_PATH = "/home/jovyan/work/tickdb_parquet"
HDFS_ROOT = "hdfs://hdfs-namenode:9000/tickdb"
OUTPUT_BASE = "/home/jovyan/work/etl_output"

df_raw = spark.read.parquet(INPUT_PATH)
print(f"Registros: {df_raw.count():,}")
df_raw.printSchema()
df_raw.show(5, truncate=False)

## 2. Transformaciones ETL

In [ ]:
df_etl = df_raw \
    .withColumn("ingestion_date", to_date(from_unixtime(col("event_timestamp") / 1000))) \
    .withColumn("ingestion_hour", hour(from_unixtime(col("event_timestamp") / 1000))) \
    .withColumn("price_range",
                when(col("last_price") < 100, "low")
                .when(col("last_price") < 1000, "medium")
                .when(col("last_price") < 10000, "high")
                .otherwise("very_high")) \
    .withColumn("volatility",
                (col("high_24h") - col("low_24h")) / col("low_24h") * 100) \
    .dropDuplicates(["symbol", "timestamp"]) \
    .cache()

print(f"Registros ETL: {df_etl.count():,}")
df_etl.select("symbol", "last_price", "market", "ingestion_date", "price_range", "volatility").show(10)

## 3. Escribir a HDFS en múltiples formatos

In [ ]:
formats = {
    "parquet": {"format": "parquet"},
    "avro": {"format": "avro"},
    "orc": {"format": "orc"},
    "delta": {"format": "delta"},
}

for name, cfg in formats.items():
    path = f"{OUTPUT_BASE}/{name}/tickdb"
    os.makedirs(path, exist_ok=True)
    df_etl \
        .repartition(2, "market") \
        .write \
        .mode("overwrite") \
        .format(cfg["format"]) \
        .partitionBy("ingestion_date", "market") \
        .option("path", path) \
        .save()
    print(f"{name.upper():8s} -> {path}")

## 4. Escribir a HDFS (cuando esté disponible)

In [ ]:
try:
    for name, cfg in formats.items():
        hdfs_path = f"{HDFS_ROOT}/{name}"
        df_etl \
            .repartition(2, "market") \
            .write \
            .mode("overwrite") \
            .format(cfg["format"]) \
            .partitionBy("ingestion_date", "market") \
            .option("path", hdfs_path) \
            .save()
        print(f"HDFS {name.upper():8s} -> {hdfs_path}")
    print("✓ HDFS escritura exitosa")
except Exception as e:
    print(f"⚠ HDFS no disponible: {e}")

## 5. Métricas de calidad

In [ ]:
print("=== CALIDAD DE DATOS ===")
total = df_etl.count()
print(f"Total registros: {total:,}")

df_etl.select(
    count("*").alias("total"),
    sum(when(col("last_price").isNull(), 1).otherwise(0)).alias("null_prices"),
    sum(when(col("symbol").isNull(), 1).otherwise(0)).alias("null_symbols"),
    sum(when(col("last_price") <= 0, 1).otherwise(0)).alias("invalid_prices"),
).show()

print("\nDistribución por mercado (particiones):")
df_etl.groupBy("market").count().orderBy("market").show()

print("\nDistribución por price_range:")
df_etl.groupBy("price_range").count().orderBy("price_range").show()

In [ ]:
spark.stop()